# DL & GenAI project milestone - 1 (23f3001252)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load datasets

In [4]:
## Load datasets
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [5]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Shape:", sample.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)
Sample Shape: (500, 2)


In [6]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [11]:
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


## Question 1
#### Calculate the frequency distribution of correct answers and the sum of the most frequent and least frequent counts.

In [8]:
# Frequency distribution
answer_freq = train['answer'].value_counts().sort_index()

print("Frequency Distribution:")
print(answer_freq)

# Most frequent and least frequent counts
most_freq = answer_freq.max()
least_freq = answer_freq.min()

result = most_freq + least_freq

print("\nMost Frequent Count:", most_freq)
print("Least Frequent Count:", least_freq)
print("Required Sum:", result)

Frequency Distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most Frequent Count: 490
Least Frequent Count: 324
Required Sum: 814


## Question 2
#### After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [10]:
import string

# Clean prompts
cleaned_prompts = (
    train['prompt']
    .str.lower()
    .str.translate(str.maketrans('', '', string.punctuation))
)

# Build vocabulary
vocab = set()

for text in cleaned_prompts:
    vocab.update(text.split())

vocab_size = len(vocab)

print("Total Unique Words (Vocabulary Size):", vocab_size)

Total Unique Words (Vocabulary Size): 859


## Question - 3
#### Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [14]:
# Get prompt where id == 1
prompt = train.loc[train['id'] == 1, 'prompt'].iloc[0]

# Lowercase and remove punctuation
cleaned_prompt = (
    prompt.lower()
    .translate(str.maketrans('', '', string.punctuation))
)

# Tokenize
words = cleaned_prompt.split()

# Remove stopwords
filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

print("Original Words:", len(words))
print("Words After Stopword Removal:", len(filtered_words))
print("Filtered Words:", filtered_words)

Original Words: 22
Words After Stopword Removal: 13
Filtered Words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


## Question - 4
#### Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine text for each row
combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

# Fit TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_text)

# Vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())

print("Number of TF-IDF Features:", vocab_size)

Number of TF-IDF Features: 2762


## Question - 5

In [21]:
## If Row ID means id == 1

from sklearn.metrics.pairwise import cosine_similarity

row = train[train['id'] == 1].iloc[0]

prompt = row['prompt']
option_a = row['A']

prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(
    prompt_vec,
    option_a_vec
)[0][0]

print("Cosine Similarity:", round(similarity, 4))

Cosine Similarity: 0.272


In [20]:
### If Row ID means dataframe index 1

row = train.iloc[1]

prompt = row['prompt']
option_a = row['A']

prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(
    prompt_vec,
    option_a_vec
)[0][0]

print(round(similarity, 4))

0.5081


## Question - 6

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

options = ['A', 'B', 'C', 'D', 'E']
correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = []

    for option in options:
        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities.append(sim)

    predicted = options[similarities.index(max(similarities))]

    if predicted == row['answer']:
        correct += 1

percentage = (correct / len(train)) * 100

print("Percentage:", round(percentage, 2))


Percentage: 13.55


## Question - 7

In [23]:
actual = 'C'
predictions = ['C', 'A', 'B']

if actual in predictions:
    score = 1 / (predictions.index(actual) + 1)
else:
    score = 0

print("MAP@3 Score:", score)


MAP@3 Score: 1.0


## Question - 8

In [24]:
actual = 'B'
predictions = ['D', 'B', 'E']

if actual in predictions:
    map3 = 1 / (predictions.index(actual) + 1)
else:
    map3 = 0

print("MAP@3:", map3)

MAP@3: 0.5


## Question - 9

In [25]:
answer_freq = train['answer'].value_counts()
print(answer_freq)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [26]:
answer_freq = train['answer'].value_counts()
top3_answers = answer_freq.index[:3].tolist()

print("Static Predictions:", top3_answers)

Static Predictions: ['B', 'C', 'A']


In [27]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)

    return 0

In [28]:
scores = []

for actual in train['answer']:
    score = apk(actual, top3_answers)
    scores.append(score)

majority_map3 = sum(scores) / len(scores)

print("Majority Class Baseline MAP@3:")
print(round(majority_map3, 4))

Majority Class Baseline MAP@3:
0.4213


## Question - 10

In [30]:
options = ['A', 'B', 'C', 'D', 'E']
scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    sims = [
        cosine_similarity(
            prompt_vec,
            vectorizer.transform([row[op]])
        )[0][0]
        for op in options
    ]

    top3 = [
        options[i]
        for i in np.argsort(sims)[::-1][:3]
    ]

    scores.append(
        apk(row['answer'], top3)
    )

tfidf_map3 = np.mean(scores)

print(f"Final MAP@3: {tfidf_map3:.4f}")


Final MAP@3: 0.2552


### Method - 2

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options = ['A', 'B', 'C', 'D', 'E']

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    sims = []

    for option in options:

        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        sims.append(sim)

    ranked_idx = np.argsort(sims)[::-1]

    top3_predictions = [
        options[i]
        for i in ranked_idx[:3]
    ]

    score = apk(
        row['answer'],
        top3_predictions
    )

    scores.append(score)

tfidf_map3 = np.mean(scores)

print("TF-IDF Pipeline MAP@3:")
print(round(tfidf_map3, 4))


TF-IDF Pipeline MAP@3:
0.2552
